## Access modes and volume modes

A PVC says how the workload wants to *use* the volume — two orthogonal axes.

### Access modes

| Mode | Code | Meaning |
|---|---|---|
| `ReadWriteOnce` | `RWO` | One **node** mounts it read-write. Default and most common; pods on that node can share it. |
| `ReadOnlyMany` | `ROX` | Many nodes mount read-only. Rare. |
| `ReadWriteMany` | `RWX` | Many nodes mount read-write. Needs NFS/CephFS/Azure Files — **not** cloud block storage (EBS, PD). |
| `ReadWriteOncePod` | `RWOP` | Only **one pod** in the whole cluster. Stronger than `RWO` (1.27+). |

A PVC requests one or more modes; the bound PV must support at least one. Here's the practical trap: cloud block storage gives `RWO`, so a **Deployment with one PVC can't easily scale to two replicas** — a second replica on a different node can't mount the same `RWO` disk. That's exactly why stateful, multi-replica systems use StatefulSets with per-pod PVCs (a few sections on).

### Volume modes

| Mode | What you get |
|---|---|
| `Filesystem` *(default)* | Formatted and mounted at `mountPath`. Files and directories. **99% of workloads.** |
| `Block` | A raw block device (`/dev/xvda`-like). For apps managing their own on-disk format. |

On our map both are properties of the **PVC** chip — the request that has to be honoured by a compatible **PV**. The CKA loves the `RWO`-can't-scale gotcha.